In [3]:
import json, shutil, random
from pathlib import Path

IMG_EXT_CANDIDATES = ['.png', '.jpg', '.jpeg', '.bmp']

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def clear_dir(p: Path):
    if p.exists():
        shutil.rmtree(p)
    p.mkdir(parents=True, exist_ok=True)

def autodetect_images(img_dir: Path):
    imgs = []
    for ext in IMG_EXT_CANDIDATES:
        imgs.extend(sorted(img_dir.glob(f"*{ext}")))
    return sorted(imgs)

def rank_items(items, mode: str, alpha: float):
    if mode == 'clip':
        scored = [(it['index'], float(it['clip_score'])) for it in items]
        scored.sort(key=lambda x: x[1], reverse=True)
        return [idx for idx,_ in scored]
    elif mode == 'dino':
        scored = [(it['index'], float(it['dino_score'])) for it in items]
        scored.sort(key=lambda x: x[1], reverse=True)
        return [idx for idx,_ in scored]
    elif mode == 'mixed':
        scored = [(it['index'], alpha*float(it['clip_score']) + (1.0 - alpha)*float(it['dino_score'])) for it in items]
        scored.sort(key=lambda x: x[1], reverse=True)
        return [idx for idx,_ in scored]
    else:  # 'random'
        idxs = [int(it['index']) for it in items]
        random.shuffle(idxs)
        return idxs

def process_split(root: Path, out: Path, split: str, include_real: bool, topk: int, sort_mode: str, alpha: float):
    split_root = root / split
    in_img_dir = split_root / 'original' / 'images'
    in_lbl_dir = split_root / 'original' / 'labels'  # 固定 labels
    synth_root = split_root / 'synth'

    out_img_dir = out / split / 'images'
    out_lbl_dir = out / split / 'labels'
    ensure_dir(out_img_dir); ensure_dir(out_lbl_dir)

    originals = autodetect_images(in_img_dir)
    print(f"[{split}] detected originals: {len(originals)} at {in_img_dir}")
    if len(originals) > 0:
        print('  e.g.:', originals[0].name, '...')

    num_o = num_s = 0

    for oimg in originals:
        stem = oimg.stem
        olbl = in_lbl_dir / f"{stem}.txt"

        if include_real:
            shutil.copy2(oimg, out_img_dir / oimg.name)
            shutil.copy2(olbl, out_lbl_dir / olbl.name)
            num_o += 1

        sdir = synth_root / stem
        sj = sdir / 'scores.json'
        with sj.open('r', encoding='utf-8') as f:
            scores = json.load(f)

        order = rank_items(scores['items'], sort_mode, alpha)
        chosen = order[:topk]

        for idx in chosen:
            name_wo_ext = f"{stem}_{int(idx):02d}"
            simg = None
            for ext in IMG_EXT_CANDIDATES:
                p = sdir / f"{name_wo_ext}{ext}"
                if p.exists():
                    simg = p
                    break
            if simg is None:
                continue
            slbl_name = f"{name_wo_ext}.txt"
            shutil.copy2(simg, out_img_dir / simg.name)
            shutil.copy2(olbl, out_lbl_dir / slbl_name)
            num_s += 1

    print(f"[{split}] originals added: {num_o}, synthetic added: {num_s}")
    print(f"Output images: {out_img_dir}")
    print(f"Output labels: {out_lbl_dir}")


In [6]:
ROOT = '../dataset'              # 输入根目录（含 train/ 与 valid/）
OUT = 'workdir'                  # 输出根目录
SPLITS = ['train', 'valid']      # 处理的划分
INCLUDE_REAL = True             # 是否包含原图到输出
TOPK = 2                         # 每张原图选择的合成图数量
SORT = 'dino'                   # 'clip' | 'dino' | 'mixed' | 'random'
ALPHA = 0                      # mixed: score = alpha*clip + (1-alpha)*dino

print('Params ->', dict(ROOT=ROOT, OUT=OUT, SPLITS=SPLITS, INCLUDE_REAL=INCLUDE_REAL, TOPK=TOPK, SORT=SORT, ALPHA=ALPHA))


Params -> {'ROOT': '../dataset', 'OUT': 'workdir', 'SPLITS': ['train', 'valid'], 'INCLUDE_REAL': True, 'TOPK': 2, 'SORT': 'dino', 'ALPHA': 0}


In [7]:
root = Path(ROOT)
out = Path(OUT)

clear_dir(out)
for split in SPLITS:
    process_split(root, out, split, INCLUDE_REAL, TOPK, SORT, ALPHA)


[train] detected originals: 80 at ..\dataset\train\original\images
  e.g.: 109_png.rf.42b6205bf83bd08baf2d693388e76906.jpg ...
[train] originals added: 80, synthetic added: 160
Output images: workdir\train\images
Output labels: workdir\train\labels
[valid] detected originals: 22 at ..\dataset\valid\original\images
  e.g.: 102_png.rf.06b2881989c28d0af7e098b419fa9e8e.jpg ...
[valid] originals added: 22, synthetic added: 44
Output images: workdir\valid\images
Output labels: workdir\valid\labels
